In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Traspilar circuit in remoto con il Qiskit Transpiler Service

> **Danger:** A partire dal 18 luglio 2025, il servizio è in fase di migrazione per supportare la nuova IBM Quantum&reg; Platform e non è disponibile. Per i pass AI, puoi utilizzare la [modalità locale](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Il servizio è una versione beta, soggetta a modifiche.
>     Se hai feedback o vuoi contattare il team di sviluppo, utilizza questo canale del [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Il Qiskit Transpiler Service fornisce funzionalità di traspirazione sul cloud. Oltre alle funzionalità del transpiler Qiskit locale, le tue attività di traspirazione possono beneficiare sia delle risorse cloud di IBM Quantum che dei pass del transpiler basati sull'AI.

Il Qiskit Transpiler Service offre una libreria Python per integrare perfettamente questo servizio e le sue funzionalità nei tuoi pattern e flussi di lavoro Qiskit esistenti. Questo servizio è disponibile esclusivamente per gli utenti dei piani IBM Quantum Premium, Flex e On-Prem (tramite IBM Quantum Platform API).

<span id="install-transpiler-service"></span>
## Installare il pacchetto qiskit-ibm-transpiler
Per utilizzare il Qiskit Transpiler Service, installa il pacchetto `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Il pacchetto esegue l'autenticazione automaticamente usando le tue [credenziali di IBM Quantum Platform](/guides/cloud-setup), in linea con come [Qiskit Runtime le gestisce](/guides/initialize-account):
- Variabile d'ambiente: `QISKIT_IBM_TOKEN`
- File di configurazione `~/.qiskit/qiskit-ibm.json` (nella sezione `default-ibm-quantum`).

*Nota*: questo pacchetto richiede Qiskit SDK v1.X.

## Opzioni di traspirazione di qiskit-ibm-transpiler
- `backend_name` (opzionale, str) — Il nome di un backend così come verrebbe specificato da QiskitRuntimeService (ad esempio, `ibm_torino`). Se impostato, il metodo di traspirazione utilizza il layout del backend specificato per l'operazione di traspirazione. Se viene impostata un'altra opzione che influisce su queste impostazioni, come `coupling_map`, le impostazioni di `backend_name` vengono sovrascritte.
- `coupling_map` (opzionale, List[List[int]]) — Una lista valida di coupling map (ad esempio, [[0,1],[1,2]]). Se impostata, il metodo di traspirazione utilizza questa coupling map per l'operazione di traspirazione. Se definita, sovrascrive qualsiasi valore specificato per `target`.
- `optimization_level` (int) — Il livello di ottimizzazione potenziale da applicare durante il processo di traspirazione. I valori validi sono [1,2,3], dove 1 corrisponde alla minima ottimizzazione (e alla massima velocità), e 3 alla massima ottimizzazione (e al maggiore dispendio di tempo).
- `ai` ("true", "false", "auto") — Indica se utilizzare le funzionalità basate sull'AI durante la traspirazione. Le funzionalità AI disponibili possono riguardare i pass di traspirazione `AIRouting` o altri metodi di sintesi basati sull'AI. Se il valore è `"true"`, il servizio applica diversi pass di traspirazione basati sull'AI in base all'`optimization_level` richiesto. Se è `"false"`, utilizza le ultime funzionalità di traspirazione di Qiskit senza AI. Infine, se è `"auto"`, il servizio decide autonomamente se applicare i pass euristici standard di Qiskit o i pass basati sull'AI in base al tuo circuit.
- `qiskit_transpile_options` (dict) — Un oggetto dizionario Python che può includere qualsiasi altra opzione valida nel [metodo `transpile()` di Qiskit](defaults-and-configuration-options). Se l'input `qiskit_transpile_options` include `optimization_level`, questo viene ignorato in favore dell'`optimization_level` specificato come parametro in ingresso. Se `qiskit_transpile_options` include un'opzione non riconosciuta dal metodo `transpile()` di Qiskit, la libreria genera un errore.

Per ulteriori informazioni sui metodi disponibili di `qiskit-ibm-transpiler`, consulta il [riferimento API di qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Per saperne di più sull'API del servizio, consulta la [documentazione REST API del Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Esempi
Gli esempi seguenti mostrano come traspilare circuit usando il Qiskit Transpiler Service con diversi parametri.

1. Crea un circuit e chiama il Qiskit Transpiler Service per traspilarlo con `ibm_torino` come `backend_name`, 3 come `optimization_level`, e senza utilizzare l'AI durante la traspirazione.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Nota*: puoi utilizzare solo i dispositivi `backend_name` a cui hai accesso con il tuo account IBM Quantum. Oltre al `backend_name`, il `TranspilerService` accetta anche `coupling_map` come parametro.

2. Produci un circuit simile e traspilalo, richiedendo le funzionalità di traspirazione AI impostando il flag `ai` su `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Produci un circuit simile e traspilalo lasciando che il servizio decida se utilizzare i pass di traspirazione basati sull'AI.